# Tools with LangGraph

In the RAG notebook we used `client.responses.create` and let the **server** handle the agent loop, i.e. the tool discovery, execution, and feeding results back to the model all happened inside one API call.

Here we move that loop to the **client** using **LangGraph**. The model and the ticketing MCP server are still the same, but now we control each step: we can inspect intermediate state, add human-in-the-loop approvals, or branch the workflow based on a tool result.  
Both patterns have different benefits, so choose the one that fits you best.

Because OGX exposes an OpenAI-compatible `/v1` endpoint, connecting LangGraph to it requires only one line to change compared to a standard OpenAI setup.

## Install dependencies

In [ ]:
import os
os.environ["PIP_INDEX_URL"] = "https://pypi.org/simple"

!pip install -q "mcp>=1.6.0,<2.0.0" langchain langchain-openai langchain-mcp-adapters langgraph

In [ ]:
import asyncio
import json
import logging
import os
import mlflow
import warnings
warnings.filterwarnings("ignore")

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

mlflow.langchain.autolog(silent=True)
logging.getLogger("mlflow.utils.autologging_utils").setLevel(logging.ERROR)


## Configuration

In [ ]:
# ──────────────────────────────────────────────────────────
# MLflow
# ──────────────────────────────────────────────────────────

# Your project namespace. Used for the MLflow workspace and the in-cluster service URLs.
with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    NAMESPACE = f.read().strip()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
EXPERIMENT_NAME = "hospital-helpdesk"
PROMPT_NAME = "ai-hospital-helpdesk"

os.environ["MLFLOW_WORKSPACE"] = NAMESPACE
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

# Only override the tracking URI when we actually have one: passing an empty value
# makes MLflow fall back to a local ./mlruns file store, which it refuses to use.
if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

experiment_id = mlflow.set_experiment(EXPERIMENT_NAME)

Like before, we need the model name here as well, if you forgot it you can find it again in AI Hub -> Deployments  
You can also leave it empty and it will automatically select the first model it finds

![Model Name](assets/model-name.png)

In [ ]:
MODEL_NAME = ""

In [ ]:
# ──────────────────────────────────────────────────────────
# Model — pointing LangChain at OGX via /v1
# ──────────────────────────────────────────────────────────

# OGX server — the in-cluster service backing the Playground in this project
OGX_URL = f"http://lsd-genai-playground-service.{NAMESPACE}.svc.cluster.local:8321"

with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as f:
    OPENSHIFT_TOKEN = f.read().strip()

OGX_HEADERS = {"X-OGX-Provider-Data": json.dumps({"passthrough_api_key": OPENSHIFT_TOKEN})}

if MODEL_NAME:
    OGX_MODEL = f"genai-bff-proxy/{MODEL_NAME}"
else:
    _ogx = OpenAI(base_url=f"{OGX_URL}/v1", api_key="unused", default_headers=OGX_HEADERS)
    OGX_MODEL = next(m.id for m in _ogx.models.list().data if m.id.startswith("genai-bff-proxy/"))

print(f"Using model: {OGX_MODEL}")

# Internal cluster URL of the MCP ticketing server (streamable-HTTP transport)
MCP_TICKETING_URL = f"http://mcp-ticketing.{NAMESPACE}.svc.cluster.local:8080/mcp"

model = ChatOpenAI(
    base_url=f"{OGX_URL}/v1",
    model=OGX_MODEL,
    api_key="not-needed",
    temperature=0.1,
    default_headers=OGX_HEADERS,
    model_kwargs={"parallel_tool_calls": False},  # model only supports one tool call at a time
)

## Load the system prompt from MLflow

Same prompt that was registered in chapter 2

In [ ]:
sys_prompt_mlflow = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")

# Prompts saved from the Playground are text prompts, so format() returns a plain string.
# Chat prompts return a list of {"role": ..., "content": ...} messages instead.
_formatted = sys_prompt_mlflow.format()
sys_prompt_plaintext = _formatted if isinstance(_formatted, str) else next(
    m["content"] for m in _formatted if m["role"] == "system")

print(sys_prompt_plaintext)

## Build the LangGraph agent

`MultiServerMCPClient` connects to the MCP server and converts its tool schemas into LangChain tools.  
`create_react_agent` builds a ReAct graph: the model node decides which tool to call, the tool node runs it, and the result loops back to the model until it has a final answer.

In [ ]:
import json as _json
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessageChunk, ToolMessage

def _content_to_str(content):
    """Flatten MCP content-block lists to a plain string the model can read."""
    if isinstance(content, list):
        return "\n".join(
            b.get("text", str(b)) if isinstance(b, dict) else str(b)
            for b in content
        )
    return str(content)


class SmartToolNode(ToolNode):
    """ToolNode that:
    - Strips 'null'/'none' string values from tool call args (small models often
      pass these for optional parameters instead of omitting them).
    - Converts MCP content-block list responses to plain strings so the model
      can read them without looping.
    - Deduplicates identical tool calls within one agent invocation so the model
      doesn't waste steps re-fetching the same data.
    """
    def __init__(self, tools):
        super().__init__(tools)
        self._cache: dict = {}  # (name, args_json) -> result string

    async def ainvoke(self, input, config=None, **kwargs):
        messages = input.get("messages", []) if isinstance(input, dict) else []
        last_ai = next(
            (m for m in reversed(messages) if hasattr(m, "tool_calls") and m.tool_calls),
            None,
        )
        injected = []

        if last_ai is not None:
            kept = []
            for tc in last_ai.tool_calls:
                tc["args"] = {
                    k: v for k, v in tc["args"].items()
                    if str(v).lower() not in ("null", "none", "")
                }
                cache_key = (tc["name"], _json.dumps(tc["args"], sort_keys=True))
                if cache_key in self._cache:
                    injected.append(ToolMessage(
                        content=self._cache[cache_key],
                        tool_call_id=tc["id"],
                        name=tc["name"],
                    ))
                else:
                    kept.append(tc)
            last_ai.tool_calls = kept

        if last_ai is not None and not last_ai.tool_calls:
            return {"messages": injected}

        result = await super().ainvoke(input, config, **kwargs)

        if isinstance(result, dict) and "messages" in result:
            for msg in result["messages"]:
                if isinstance(msg, ToolMessage):
                    msg.content = _content_to_str(msg.content)
                    if last_ai is not None:
                        for tc in last_ai.tool_calls:
                            if tc["id"] == msg.tool_call_id:
                                key = (tc["name"], _json.dumps(tc["args"], sort_keys=True))
                                self._cache[key] = msg.content
            result["messages"] = injected + result["messages"]

        return result


_TOOL_GUIDANCE = """

When using tools:
- Call tools through the tool interface — never write tool calls as raw JSON in your response.
- Never guess, invent, or modify ticket IDs. Only use IDs that appear verbatim in the user's message or in a tool result. If you receive an error saying a ticket was not found, stop and tell the user — do not try alternative IDs or create a new ticket.
- Omit optional parameters — never pass null, 'null', or 'none'.
- Once you receive a tool result that answers the question, stop and give your answer. Do not make additional tool calls just to verify what you already know.
- Only take actions (create, update) when the user explicitly asks for them. Never modify tickets unless directly instructed to do so.
"""

async def ask(message: str, stream_output: bool = True) -> dict:
    """Run one question through the LangGraph agent.

    With stream_output=True the model's text is printed as it is generated.
    """
    mcp_client = MultiServerMCPClient(
        {"ticketing": {"transport": "streamable_http", "url": MCP_TICKETING_URL}}
    )
    raw_tools = await mcp_client.get_tools()
    tool_node = SmartToolNode(raw_tools)
    agent = create_react_agent(model, tool_node, prompt=sys_prompt_plaintext + _TOOL_GUIDANCE)

    # Wrap in a named span so the MLflow trace title shows the user question.
    # autolog fills in the child spans (model calls, tool calls) automatically.
    with mlflow.start_span(name=message) as span:
        span.set_inputs(message)
        # stream_mode=["messages", "values"] gives us two things at once:
        # "messages" yields each token as the model produces it, and
        # "values" yields the full graph state so we still get the final answer.
        result = None
        async for mode, payload in agent.astream(
            {"messages": [("user", message)]},
            config={"recursion_limit": 10},
            stream_mode=["messages", "values"],
        ):
            if mode == "messages":
                chunk, _meta = payload
                text = chunk.content if isinstance(chunk, AIMessageChunk) else ""
                if text and stream_output:
                    print(text.replace("\n", "\n  "), end="", flush=True)
            else:
                result = payload

        if stream_output:
            print()

        answer = result["messages"][-1].content
        span.set_outputs(answer)

    # Extract tool steps for display
    tool_steps = []
    pending_calls = {}
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                pending_calls[tc["id"]] = {"name": tc["name"], "args": tc["args"]}
        if msg.type == "tool":
            call = pending_calls.pop(msg.tool_call_id, {"name": msg.name, "args": {}})
            tool_steps.append({"name": call["name"], "args": call["args"], "result": _content_to_str(msg.content)})

    return {"answer": answer, "tool_steps": tool_steps}


## Try it

These four questions exercise different tools on the server.  
Notice that the intermediate tool calls are now printed in the cell output — because the loop runs here, not on the server.

In [ ]:
messages = [
    "Can you check the status of ticket TKT-002 for me?",
    "What open tickets do we currently have?",
    "A staff member called Mark Davies on Ward 6C says his network connection has been dropping every hour since this morning. Can you raise a ticket for him?",
    "Please mark ticket TKT-003 as resolved and add a comment that the paper jam was cleared.",
]

In [ ]:
width = 100

for message in messages:
    print(f"\n{'━' * width}")
    print(f"  Q: {message}")
    print(f"{'━' * width}")

    # The answer streams in as it is generated
    print(f"  💬 Answer:")
    print("  ", end="")
    result = await ask(message)
    tool_steps = result["tool_steps"]

    if tool_steps:
        print(f"\n  🔧 Tool calls ({len(tool_steps)}):")
        for step in tool_steps:
            args_str = ", ".join(f"{k}={repr(v)}" for k, v in step["args"].items())
            result_preview = str(step["result"]).replace("\n", " ").strip()
            if len(result_preview) > 200:
                result_preview = result_preview[:197] + "..."
            print(f"  ┌─ {step['name']}({args_str})")
            print(f"  │  {result_preview}")

    print(f"{'━' * width}\n")
